In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [1]:
import sys

import torch

from spice import SpiceEstimator

sys.path.append("../../..")
from weinhardt2026.utils.benchmarking_gru import GRUModel, training
from weinhardt2026.studies.castro2025.benchmarking_castro2025 import Castro2025Model, get_dataset, generate_behavior
import spice_castro2025, spice_castro2025_2

## Load dataset

In [2]:
path_data = 'data/eckstein2024.csv'
test_sessions = (2,)  # pick sessions that exist for all participants; adjust if needed
dataset_train, dataset_test, info_dataset = get_dataset(path_data=path_data, test_sessions=test_sessions, verbose=True)

Shape of dataset: torch.Size([4158, 150, 1, 13])
Number of participants: 862
Number of actions in dataset: 4


In [3]:
from spice import SpiceDataset

# keep only 100 timesteps
dataset_train = SpiceDataset(dataset_train.xs[:, :100], dataset_train.ys[:, :100])

# keep only 100 participants for rapid prototyping
keep_participants = torch.arange(0, 50)

def keep_subset(dataset, subset):
    participant_ids = dataset.xs[:, 0, 0, -1]
    mask = torch.isin(participant_ids, subset)
    return SpiceDataset(dataset.xs[mask], dataset.ys[mask])

dataset_train = keep_subset(dataset_train, keep_participants)
dataset_test = keep_subset(dataset_test, keep_participants)    


## SPICE Setup

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [ ]:
# fitting without SINDy coefficients
path_spice = 'params/spice_castro2025_2.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025_2.SpiceModel,
        spice_config=spice_castro2025_2.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1,
        
        sindy_weight=0,
        sindy_library_polynomial_degree=2,
        sindy_pruning_frequency=100,
        sindy_ensemble_pruning=0.05,
        sindy_threshold_pruning=0.01,
        sindy_alpha=0.0001,
        
        save_path_spice=path_spice,
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [18]:
path_spice = 'params/spice_castro2025_2.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025_2.SpiceModel,
        spice_config=spice_castro2025_2.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1000,
        warmup_steps=200,
        learning_rate=0.01,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,

        verbose=True,
        # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

 24%|████████▉                            | 243/1000 [19:54<1:02:01,  4.92s/it, L(Train)=0.4903638, L(Val,RNN)=0.5158772, L(Val,SINDy)=0.8586412, Conv=1.40e-03]
----------------------------------------------------------------------------------------------------------------------------------------------------------------
SPICE Model (Coefficients: 37):
value_reward_env[t+1]        = -0.04 1 + 0.961 value_reward_env[t] + 0.001 reward[t] + -0.005 value_reward_env^2 + -0.013 value_reward_env*reward[t] + 0.005 reward[t]^2 
value_reward_chosen[t+1]     = -0.377 1 + 1.005 value_reward_chosen[t] + 0.064 reward_env + 0.223 reward[t] + -0.005 value_reward_chosen^2 + 0.02 value_reward_chosen*reward_env + -0.149 value_reward_chosen*reward[t] + -0.005 reward_env^2 + -0.122 reward_env*reward[t] + 0.453 reward[t]^2 
value_reward_not_chosen[t+1] = -0.008 1 + 0.985 value_reward_not_chosen[t] + 0.014 reward_env + -0.003 value_reward_not_chosen^2 + 0.002 value_reward_not_chosen*reward_env + 0.001 reward_

  6%|▌         | 55/1000 [00:18<06:09,  2.56it/s, loss=0.0244261, n_params=37.00+/-0.00]

In [19]:
estimator.load_spice(path_spice)
# estimator.aggregate_coefficients()

In [20]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=0)


Example SPICE model (participant 0):
value_reward_env[t+1]        = 0.221 1 + 0.945 value_reward_env[t] + 0.024 reward[t] + -0.011 value_reward_env^2 + -0.039 value_reward_env*reward[t] + 0.018 reward[t]^2 
value_reward_chosen[t+1]     = -0.189 1 + 0.945 value_reward_chosen[t] + -0.072 reward_env + 0.26 reward[t] + 0.021 value_reward_mean + -0.003 value_reward_chosen^2 + -0.012 value_reward_chosen*reward_env + -0.124 value_reward_chosen*reward[t] + 0.02 value_reward_chosen*value_reward_mean + -0.016 reward_env^2 + 0.145 reward_env*reward[t] + 0.011 reward_env*value_reward_mean + 0.521 reward[t]^2 + -0.107 reward[t]*value_reward_mean + -0.018 value_reward_mean^2 
value_reward_not_chosen[t+1] = -0.274 1 + 0.796 value_reward_not_chosen[t] + -0.174 reward_env + 0.148 value_reward_mean + -0.008 value_reward_not_chosen^2 + -0.131 value_reward_not_chosen*reward_env + 0.091 value_reward_not_chosen*value_reward_mean + -0.079 reward_env^2 + 0.052 reward_env*value_reward_mean + -0.026 value_rewa

## Benchmarking

### Castro2025 benchmark model

In [23]:
path_spice = 'params/spice_castro2025.pkl'

# Benchmark model: Castro et al. 2025
cfs = Castro2025Model(
    n_participants=dataset_train.n_participants,
    n_actions=dataset_train.n_actions,
    batch_first=True,
    )

path_cfs = path_spice.replace('spice_', 'cfs_')

In [ ]:
optimizer_cfs = torch.optim.Adam(params=cfs.parameters(), lr=0.01)
epochs = 1000

cfs = training(
    model=cfs,
    optimizer=optimizer_cfs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
)

torch.save(cfs.state_dict(), path_cfs)

In [12]:
cfs.load_state_dict(torch.load(path_cfs, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [13]:
gru = GRUModel(
    n_actions=dataset_train.n_actions, 
    additional_inputs=2, 
    dropout=0.1,
    hidden_size=32,
    )
path_gru = path_spice.replace('spice_', 'gru_')

In [9]:
epochs = 1000
optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer_gru,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

Epoch 1/1000: L(Train): 1.4684736728668213; L(Test): 1.5077646970748901
Epoch 2/1000: L(Train): 1.4892171621322632; L(Test): 1.4341553449630737
Epoch 3/1000: L(Train): 1.4499824047088623; L(Test): 1.3870218992233276
Epoch 4/1000: L(Train): 1.399340033531189; L(Test): 1.3862683773040771
Epoch 5/1000: L(Train): 1.3948732614517212; L(Test): 1.389448642730713
Epoch 6/1000: L(Train): 1.3978655338287354; L(Test): 1.3850520849227905
Epoch 7/1000: L(Train): 1.3922415971755981; L(Test): 1.3773967027664185
Epoch 8/1000: L(Train): 1.3848704099655151; L(Test): 1.3699711561203003
Epoch 9/1000: L(Train): 1.377264142036438; L(Test): 1.3622416257858276
Epoch 10/1000: L(Train): 1.3698391914367676; L(Test): 1.3585764169692993
Epoch 11/1000: L(Train): 1.365770936012268; L(Test): 1.3521857261657715
Epoch 12/1000: L(Train): 1.3602874279022217; L(Test): 1.3406612873077393
Epoch 13/1000: L(Train): 1.350714087486267; L(Test): 1.331235408782959
Epoch 14/1000: L(Train): 1.3407236337661743; L(Test): 1.3191748857

In [14]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [15]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from analysis_generative import analysis_generative_behavior

## General Analysis

In [15]:
# Dataset-specific behavioral analysis placeholder.
# Replace with Eckstein2024-specific columns if needed.

In [16]:
print("Fitted Castro2025 parameters:")
print("\nBeta_r")
print(cfs.beta_r)
print("\nLapse")
print(cfs.lapse)
print("\nPrior")
print(cfs.prior)
print("\nAlpha exploration rate")
print(cfs.alpha_exploration_rate)
print("\nDecay rate")
print(cfs.decay_rate)
print("\nGamma")
print(cfs.gamma)
print("\nTemperature")
print(cfs.temperature)

Fitted Castro2025 parameters:

Beta_r
tensor([2.6588, 2.5872, 2.4406, 2.6166, 1.5054, 2.2513, 2.1100, 2.1039, 1.3746,
        2.0745, 2.6667, 2.1328, 2.3358, 2.6025, 2.1986, 2.1311, 2.8331, 2.4313,
        2.6382, 3.0562, 2.2559, 2.3994, 1.6902, 2.9913, 1.8730, 1.9825, 1.6819,
        2.4229, 2.0172, 2.3439, 1.9366, 2.0212, 2.0829, 2.6930, 2.3627, 2.4913,
        2.2294, 2.5848, 2.2695, 1.6097, 2.0131, 1.8442, 2.4099, 1.7877, 2.4884,
        2.5506, 2.2508, 1.9150, 2.4284, 2.4954, 2.2797, 1.9747, 2.2009, 1.3223,
        1.7321, 1.8294, 1.8130, 2.4777, 2.4677, 2.4375, 1.7650, 2.0791, 2.4099,
        1.9940, 2.5261, 1.7374, 2.3255, 2.0326, 1.4980, 1.6791, 1.8890, 1.9227,
        3.0623, 3.0400, 1.6810, 2.5141, 2.1057, 1.5235, 2.4788, 3.1193, 2.7911,
        2.1080, 2.4988, 1.9093, 1.6636, 2.5500, 1.7713, 2.7464, 1.7287, 2.7249,
        1.5610, 2.2452, 2.2940, 3.5569, 1.9799, 1.0523, 2.3127, 1.8893, 2.3050,
        2.3087, 2.0803, 1.8150, 2.6862, 2.3649, 2.2447, 2.5190, 1.9997, 2.4064,
  

## Analysis Model Evaluation

In [21]:
analysis_model_evaluation(
    dataset=dataset_train,
    spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.656590,0.159280,13.000000,0.000000,7890.157227,15806.314453,15908.224609
GRU,0.658189,0.152148,6852.000000,0.000000,7844.539062,29393.078125,83107.382812
SPICE-RNN,0.615619,0.158915,10614.000000,0.000000,9098.567383,39425.132812,122630.562500
SPICE,0.614896,0.158621,54.720001,0.448509,9120.589844,18350.626953,18779.615234


In [22]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.638818,0.176031,13.000000,0.000000,3323.375977,6672.751953,6762.600098
GRU,0.653981,0.164939,6852.000000,0.000000,3149.399658,20002.798828,67359.679688
SPICE-RNN,0.607736,0.175655,10614.000000,0.000000,3693.276367,28614.552734,101972.101562
SPICE,0.606805,0.175069,54.720001,0.453557,3704.651611,7518.743164,7896.934570


## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

## Analysis Individual Differences

In [ ]:
# analysis_coefficients_individuals(
#     criterion="SomeCriterionColumnInYourDataset",
#     analysis="disc",  # also: "cont"
#     reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"
    
#     path_data=path_file,
    
#     spice_model=estimator,
    
#     dir_output='results',
# )

## Generative Behavior Analysis

In [ ]:
estimator.eval()
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice.csv'
)

estimator.use_sindy(False)
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice_rnn.csv'
)

gru.eval()
generate_behavior(
    model=gru,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_gru.csv'
)

generate_behavior(
    model=cfs,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_cfs.csv'
)

In [ ]:
analysis_generative_behavior(
    path_data_real=path_data,
    path_data_gru='data/eckstein2024_gru.csv',
    path_data_benchmark='data/eckstein2024_cfs.csv',
    path_data_spice='data/eckstein2024_spice.csv',
    path_data_spice_rnn='data/eckstein2024_spice_rnn.csv',
    output_dir='results',
)